# Fase 2: Limpieza y normalización textual
---
Este cuaderno documenta el proceso de limpieza semántica y normalización de las letras de canciones obtenidas en la fase anterior. Dado que el contenido extraído mediante web scraping contiene metadatos "impuros", es imprescindible preparar un *corpus* de alta calidad antes de inyectarlo en los modelos de lenguaje.

El objetivo es eliminar versiones redundantes, limpiar las anotaciones estructurales y aplicar técnicas de procesamiento de lenguaje natural (como la lematización y la eliminación de stopwords) mediante la librería `spaCy`.

### 2.1 Importación de librerías y modelos de procesamiento de lenguaje natural
Se carga la base de datos cruda y el modelo lingüístico `en_core_web_sm` de `spaCy`, optimizado para el análisis morfológico del idioma inglés.

In [2]:
import pandas as pd
import re
import spacy

In [3]:
try:
    nlp = spacy.load("en_core_web_sm")
except:
    print("Modelo de spaCy no encontrado. Por favor, instálalo usando: python -m spacy download es_core_web_sm")

In [4]:
df = pd.read_csv("../data/interim/taylor_swift_discography.csv")
print(f"Canciones en el dataset: {len(df)}")

Canciones en el dataset: 250


### 2.2 Filtrado de ruido y versiones alternativas
Las discografías modernas suelen incluir versiones que introducen sesgos y duplicidades indeseadas en el recuento de vocabulario o en el análisis de sentimientos. Se aplica una *blacklist* para purgar remixes, versiones acústicas y prólogos, manteniendo únicamente las canciones canónicas del catálogo.

In [5]:
blacklist = ["Remix", "Piano Version", "Acoustic Version", "Prologue"]

def is_noise(title):
    for keyword in blacklist:
        if keyword.lower() in title.lower():
            return True
    return False

mask = df['title'].apply(is_noise)
dropped_songs = df[mask]

print("\n" + "-"*30)
print(f"Eliminando {len(dropped_songs)} canciones:")
print("-"*30)

for title in dropped_songs['title'].tolist():
    print(f" - {title}")
print("-"*30 + "\n")

df_clean = df[~mask].copy()

print(f"Canciones tras eliminar ruido: {len(df_clean)}")


------------------------------
Eliminando 8 canciones:
------------------------------
 - Reputation [Prologue]
 - Forever & Always (Piano Version) [Taylor’s Version]
 - Love Story (Taylor’s Version) [Elvira Remix]
 - State Of Grace (Acoustic Version) (Taylor’s Version)
 - 1989 (Taylor’s Version) [Prologue]
 - Snow on the Beach (Remix)
 - Karma (Remix)
 - Speak Now (Taylor’s Version) [Prologue]
------------------------------

Canciones tras eliminar ruido: 242


### 2.3 Limpieza estructural
Los textos extraídos de Genius contienen etiquetas que indican la estructura de la canción (p.ej. `[Chorus]`, `[Verse 1]`). Utilizando expresiones regulares, se eliminan estos corchetes y se normalizan los saltos de línea para evitar que los algoritmos de inteligencia artificial interpreten estos metadatos como parte de la letra.

In [6]:
def clean_lyrics(text):
    if not isinstance(text, str):
        return ""
    
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\n+', '\n', text)

    return text

df_clean['lyrics_full'] = df_clean['lyrics'].apply(clean_lyrics)

### 2.4 Compresión narrativa y eliminación de repeticiones
La música pop tiende a la reiteración excesiva (p.ej. "haters gonna hate, hate, hate"). Para no alterar negativamente las métricas de riqueza léxica ni sobrecargar el contexto de los LLM, se ha desarrollado una función que identifica y colapsa los bucles de repeticiones utilizando expresiones regulares, y posteriormente elimina las líneas que son idénticas.

In [7]:
def get_unique_lines(text):
    if not isinstance(text, str):
        return ""
    
    lines = text.split('\n')
    cleaned_lines = []
    
    for line in lines:
        line = re.sub(r'\b(\w+)(?:[\s,]+\1\b)+', r'\1', line, flags=re.IGNORECASE)
        line = re.sub(r'(\b.+?)(?:,\s+\1)+', r'\1', line, flags=re.IGNORECASE)

        cleaned_lines.append(line)
    
    unique_lines = list(dict.fromkeys([line for line in cleaned_lines if line]))
    return "\n".join(unique_lines)

df_clean['lyrics_unique'] = df_clean['lyrics_full'].apply(get_unique_lines)

In [8]:
test_phrase = "And the haters gonna hate, hate, hate"
print(f"Test: '{test_phrase}' -> '{get_unique_lines(test_phrase)}'")

Test: 'And the haters gonna hate, hate, hate' -> 'And the haters gonna hate'


### 2.5 Normalización lingüística
Para las futuras fases de modelado de temas y cálculo de complejidad, se ha generado una versión paralela del texto. Utilizando el procesamiento de `spaCy`, se convirtió cada palabra a su raíz morfológica y se filtraron stopwords y signos de puntuación.

In [9]:
def lemmatize_text(text):
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.is_space]
    return " ".join(tokens)

df_clean['lyrics_norm'] = df_clean['lyrics_unique'].apply(lemmatize_text)

### 2.6 Exportación de los datos
Con el texto estructurado en distintos niveles, se volcaron los resultados a un archivo `_processed.csv`.

In [10]:
output_file = "taylor_swift_processed.csv"
df_clean.to_csv(f"../data/processed/{output_file}", index=False)

print("\n" + "="*50)
print(f"Dataset procesado: {output_file}\nTotal canciones: {len(df_clean)}")
print("="*50)


Dataset procesado: taylor_swift_processed.csv
Total canciones: 242


In [11]:
print("Ejemplo de Título:", df_clean.iloc[0]['title'])
print("Ejemplo de Letra:", df_clean.iloc[0]['lyrics_full'][:50].replace('\n', ' '))

Ejemplo de Título: Tim McGraw
Ejemplo de Letra:  He said the way my blue eyes shined Put those Geo
